## Notebook summary

| Item | Details |
| --- | --- |
| Purpose | 03 - SE-ResNeXt-50 Paired-View YOLO Adaptation Training (384) |
| Model | SE-ResNeXt-50 32x4d |
| Input | 384x384 paired (published 224 + YOLO ROI 224) |
| Task | Paired-view YOLO ROI adaptation training |
| Loss | Cross-Entropy (CE) |
| Configuration | batch 48, lr 1e-5, 5 epochs, full model unfrozen |
| Result | See the executed cells below for metrics, plots, and checkpoint details. |
| Status | training notebook |

Mirrors DenseNet121 `03_train_densenet121_paired_view_yolo_384.ipynb` exactly:
- published 224x224 crop: used 50% of the time
- expanded 1.15x YOLO square ROI: used 50% of the time
- five fine-tuning epochs from the CE SE-ResNeXt-50 checkpoint
- CE is the only training loss; no MSE or ordinal loss is introduced
- Validation reports both source views and selects the checkpoint by their mean selection score
- Test data is not used for tuning

# 03 - SE-ResNeXt-50 Paired-View YOLO Adaptation Training (384)

Run notebook 02 first. This is the SE-ResNeXt-50 counterpart of the selected
DenseNet121 paired-view adaptation, with every data and optimization choice
held constant:

- published 224x224 crop: used 50% of the time
- expanded 1.15x YOLO square ROI: used 50% of the time
- five fine-tuning epochs from the selected CE SE-ResNeXt-50 checkpoint (from notebook 02)
- CE is the only training loss; no MSE or ordinal loss is introduced
- All layers unfrozen from the start (single-stage, no freeze/unfreeze)
- Validation reports both source views and selects the checkpoint by their mean selection score
- Test data is not used for tuning

In [1]:
!pip -q install "timm>=1.0" "h5py>=3.9"

In [2]:
from google.colab import drive
drive.mount("/content/drive")

import json
import random
from datetime import datetime, timezone
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import average_precision_score, cohen_kappa_score, precision_recall_fscore_support
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms
from tqdm.auto import tqdm

Mounted at /content/drive


## Fixed paired-view configuration

Mirrors DenseNet121 03_: full model unfrozen, single lr, no stage switching.

In [3]:
SEED = 42
INPUT_SIZE = 384
# A100 profile: 64 fits a 40-GB card; 128 is used on an 80-GB card.
GPU_MEMORY_GB = (torch.cuda.get_device_properties(0).total_memory / 2**30) if torch.cuda.is_available() else 0.0
IS_A100 = torch.cuda.is_available() and "A100" in torch.cuda.get_device_name(0)
BATCH_SIZE = 128 if IS_A100 and GPU_MEMORY_GB >= 70 else (64 if IS_A100 and GPU_MEMORY_GB >= 35 else 48)
NUM_WORKERS = 8 if IS_A100 else 2
EPOCHS = 5
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 1e-3
ALTERNATE_VIEW_PROBABILITY = 0.50

PUBLISHED_ROOT = Path(
    "/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/"
    "extracted/KneeXrayData/ClsKLData/kneeKL224"
)
ROI_ROOT = Path(
    "/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/"
    "derived/densenet121_yolo_square_roi_trainvaltest_v2"
)
BASE_CHECKPOINT_ROOT = Path("/content/drive/MyDrive/Models/seresnext50_32x4d_checkpoints")
# Notebook 02 writes this pointer after selecting its best 3-stage checkpoint.
# Reading it avoids silently reusing an old hard-coded run timestamp.
selected_pointers = sorted(
    BASE_CHECKPOINT_ROOT.glob("*/SELECTED_CHECKPOINT.txt"),
    key=lambda path: path.stat().st_mtime, reverse=True,
)
if not selected_pointers:
    raise FileNotFoundError(
        f"No Notebook 02 checkpoint pointer found under {BASE_CHECKPOINT_ROOT}. "
        "Run 02_train_se_resnext50_original_384 first."
    )
BASE_CHECKPOINT = Path(selected_pointers[0].read_text().strip())
RUN_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%d_%H-%M-%S_%f_UTC")
RUN_DIR = Path("/content/drive/MyDrive/Models/seresnext50_32x4d_yolo_384") / RUN_TIMESTAMP

for required in (PUBLISHED_ROOT, ROI_ROOT, BASE_CHECKPOINT):
    if not required.exists():
        raise FileNotFoundError(required)
RUN_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

Device: cuda


## Build paired published/YOLO records

Every published image must have the same patient-side file in the generated ROI folder.
The test split is deliberately excluded from adaptation and checkpoint selection.

In [4]:
rows = []
for split in ("train", "val"):
    for grade in range(5):
        for published_path in sorted((PUBLISHED_ROOT / split / str(grade)).glob("*.png")):
            roi_path = ROI_ROOT / split / str(grade) / published_path.name
            if not roi_path.is_file():
                raise FileNotFoundError(f"Missing paired ROI: {roi_path}")
            rows.append({
                "split": split,
                "grade": grade,
                "published_path": str(published_path),
                "roi_path": str(roi_path),
            })
frame = pd.DataFrame(rows)
print(frame.groupby(["split", "grade"]).size().unstack(fill_value=0))

grade     0     1     2    3    4
split                            
train  2286  1046  1516  757  173
val     328   153   212  106   27


## Preprocessing, paired dataset, and model

The alternate ROI is selected independently for each training sample. This is the
paired-view adaptation mechanism. It is not MSE feature matching.

In [5]:
class OpenCVCLAHE:
    def __call__(self, image_rgb):
        lab = cv2.cvtColor(np.asarray(image_rgb), cv2.COLOR_RGB2LAB)
        lightness, a, b = cv2.split(lab)
        lightness = cv2.createCLAHE(clipLimit=1.25, tileGridSize=(8, 8)).apply(lightness)
        return cv2.cvtColor(cv2.merge((lightness, a, b)), cv2.COLOR_LAB2RGB)


class SquarePad:
    def __call__(self, image_rgb):
        image = np.asarray(image_rgb)
        height, width = image.shape[:2]
        side = max(height, width)
        top, left = (side - height) // 2, (side - width) // 2
        return cv2.copyMakeBorder(
            image, top, side - height - top,
            left, side - width - left,
            cv2.BORDER_CONSTANT, value=(0, 0, 0)
        )

train_transform = transforms.Compose([
    OpenCVCLAHE(), SquarePad(), transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(p=0.50),
    transforms.RandomRotation(5),
    transforms.ColorJitter(brightness=0.08, contrast=0.08),
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.10, scale=(0.02, 0.05), ratio=(0.5, 2.0), value=0),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    OpenCVCLAHE(), SquarePad(), transforms.ToPILImage(),
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


class PairedDataset(Dataset):
    def __init__(self, data, transform, alternate_probability):
        self.data = data.reset_index(drop=True)
        self.transform = transform
        self.alternate_probability = alternate_probability
        self.labels = self.data.grade.astype(int).tolist()

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        row = self.data.iloc[index]
        use_roi = self.alternate_probability > 0 and random.random() < self.alternate_probability
        image = cv2.imread(row.roi_path if use_roi else row.published_path, cv2.IMREAD_COLOR)
        if image is None:
            raise IOError(f"Cannot read paired image at index {index}")
        return self.transform(cv2.cvtColor(image, cv2.COLOR_BGR2RGB)), int(row.grade)


class SEResNeXt50Model(nn.Module):
    """SE-ResNeXt-50 with a linear head; structurally identical to the 02_ SEResNeXt50GradCAM
    used to produce the BASE_CHECKPOINT, so the saved classifier.weight/bias keys match."""

    def __init__(self, drop_rate=0.20):
        super().__init__()
        self.backbone = timm.create_model(
            "seresnext50_32x4d",
            pretrained=False,
            features_only=True,
            out_indices=(4,),
        )
        channels = self.backbone.feature_info.channels()[0]
        self.classifier = nn.Linear(channels, 5)

    def forward(self, images):
        features = self.backbone(images)[0]
        return self.classifier(features.mean(dim=(2, 3)))

## Fine-tune for five epochs with cross-entropy

The paired-view source experiment uses `F.cross_entropy`. Its robustness comes from
alternating views and validating on both domains, not from an MSE term.

In [6]:
checkpoint = torch.load(BASE_CHECKPOINT, map_location=DEVICE, weights_only=False)
if checkpoint.get("loss_type") not in (None, "ce"):
    raise RuntimeError(f"Expected CE checkpoint, got {checkpoint.get('loss_type')}")

model = SEResNeXt50Model(drop_rate=0.20).to(DEVICE)
model.load_state_dict(checkpoint["model_state_dict"], strict=True)

print(f"Loaded baseline: {BASE_CHECKPOINT}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,} (all layers unfrozen)")

train_frame = frame[frame.split == "train"].reset_index(drop=True)
val_frame   = frame[frame.split == "val"  ].reset_index(drop=True)
counts = np.bincount(train_frame.grade.to_numpy(), minlength=5)
weights = (1.0 / counts)[train_frame.grade.to_numpy()]
sampler = WeightedRandomSampler(
    torch.as_tensor(weights, dtype=torch.double), len(weights), replacement=True
)
train_loader = DataLoader(
    PairedDataset(train_frame, train_transform, ALTERNATE_VIEW_PROBABILITY),
    batch_size=BATCH_SIZE, sampler=sampler,
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=NUM_WORKERS > 0,
)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-7)
scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")


def evaluate(data, use_roi):
    loader = DataLoader(
        PairedDataset(data, val_transform, 1.0 if use_roi else 0.0),
        batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True,
        persistent_workers=NUM_WORKERS > 0,
    )
    labels, probabilities = [], []
    model.eval()
    with torch.inference_mode():
        for images, batch_labels in loader:
            probs = F.softmax(
                model(images.to(DEVICE, non_blocking=True)).float(), dim=1
            ).cpu().numpy()
            labels.extend(batch_labels.numpy())
            probabilities.extend(probs)
    labels = np.asarray(labels)
    probabilities = np.asarray(probabilities)
    predictions = probabilities.argmax(axis=1)
    _, _, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="macro", zero_division=0
    )
    ap = average_precision_score(np.eye(5)[labels], probabilities, average="macro")
    qwk = cohen_kappa_score(labels, predictions, weights="quadratic")
    return {
        "qwk": float(qwk),
        "macro_f1": float(f1),
        "macro_ap": float(ap),
        "selection": float(0.55 * qwk + 0.30 * f1 + 0.15 * ap),
    }


best_score = -float("inf")
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    loss_sum, samples = 0.0, 0
    for images, labels in tqdm(train_loader, desc=f"epoch {epoch}/{EPOCHS}"):
        images, labels = images.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=DEVICE.type == "cuda"):
            loss = F.cross_entropy(model(images), labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        loss_sum += loss.item() * len(labels)
        samples += len(labels)
    scheduler.step()

    published = evaluate(val_frame, use_roi=False)
    roi       = evaluate(val_frame, use_roi=True)
    robust    = 0.5 * (published["selection"] + roi["selection"])
    row = {
        "epoch": epoch,
        "train_loss": loss_sum / samples,
        "robust_selection": robust,
        **{f"published_{k}": v for k, v in published.items()},
        **{f"roi_{k}":       v for k, v in roi.items()},
    }
    history.append(row)
    print(json.dumps(row, indent=2))

    if robust > best_score:
        best_score = robust
        torch.save({
            "model_state_dict": model.state_dict(),
            "architecture": "seresnext50_32x4d_linear_gradcam",
            "loss_type": "ce",
            "epoch": epoch,
            "paired_view_probability": ALTERNATE_VIEW_PROBABILITY,
            "roi_expansion": 1.15,
            "robust_selection": robust,
        }, RUN_DIR / "best_model.pth")
        print(f"  => New best! robust_selection={robust:.4f}")

pd.DataFrame(history).to_csv(RUN_DIR / "history.csv", index=False)
(RUN_DIR / "run_config.json").write_text(json.dumps({
    "loss": "cross_entropy",
    "mse_used": False,
    "alternate_view_probability": ALTERNATE_VIEW_PROBABILITY,
    "roi_expansion": 1.15,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "input_size": INPUT_SIZE,
    "base_checkpoint": str(BASE_CHECKPOINT),
}, indent=2))
print(f"\nTraining complete. Best robust_selection: {best_score:.4f}")
print(f"Best checkpoint: {RUN_DIR / 'best_model.pth'}")

Loaded baseline: /content/drive/MyDrive/Models/seresnext50_32x4d_checkpoints/2026-08-14_17-09-16_448541_UTC_original_224_ce_3stage/best_model.pth
Total parameters: 25,521,141
Trainable: 25,521,141 (all layers unfrozen)


epoch 1/5:   0%|          | 0/91 [00:00<?, ?it/s]

{
  "epoch": 1,
  "train_loss": 0.7407016668323003,
  "robust_selection": 0.6838498881904231,
  "published_qwk": 0.7839048982386257,
  "published_macro_f1": 0.6527025656576491,
  "published_macro_ap": 0.7066792080526751,
  "published_selection": 0.7329603449364401,
  "roi_qwk": 0.6831574452510218,
  "roi_macro_f1": 0.5553378510978866,
  "roi_macro_ap": 0.6160098748465201,
  "roi_selection": 0.634739431444406
}
  => New best! robust_selection=0.6838


epoch 2/5:   0%|          | 0/91 [00:00<?, ?it/s]

{
  "epoch": 2,
  "train_loss": 0.6877664012164548,
  "robust_selection": 0.6927850012606283,
  "published_qwk": 0.794528838588634,
  "published_macro_f1": 0.665881347574788,
  "published_macro_ap": 0.7129471344592677,
  "published_selection": 0.7436973356650752,
  "roi_qwk": 0.6874189364461738,
  "roi_macro_f1": 0.5647210320452432,
  "roi_macro_ap": 0.6291729479814178,
  "roi_selection": 0.6418726668561813
}
  => New best! robust_selection=0.6928


...

{
  "epoch": 3,
  "train_loss": 0.6784918697163406,
  "robust_selection": 0.6871935149697865,
  "published_qwk": 0.7879569892473118,
  "published_macro_f1": 0.6463363110101183,
  "published_macro_ap": 0.7091174427347067,
  "published_selection": 0.733644853799263,
  "roi_qwk": 0.6802206657162339,
  "roi_macro_f1": 0.5721369647175243,
  "roi_macro_ap": 0.633198137207495,
  "roi_selection": 0.6407421761403101
}


epoch 4/5:   0%|          | 0/91 [00:00<?, ?it/s]

{
  "epoch": 4,
  "train_loss": 0.6599291929450981,
  "robust_selection": 0.6976079798172516,
  "published_qwk": 0.7912733637557042,
  "published_macro_f1": 0.6654775768638027,
  "published_macro_ap": 0.7153422378260059,
  "published_selection": 0.742144958798679,
  "roi_qwk": 0.6935232027980763,
  "roi_macro_f1": 0.5843468296542816,
  "roi_macro_ap": 0.6421946026706518,
  "roi_selection": 0.6530710008358241
}
  => New best! robust_selection=0.6976


epoch 5/5:   0%|          | 0/91 [00:00<?, ?it/s]

{
  "epoch": 5,
  "train_loss": 0.6678233751379193,
  "robust_selection": 0.6938336364868929,
  "published_qwk": 0.7868042256131463,
  "published_macro_f1": 0.6586617924252262,
  "published_macro_ap": 0.7139460495364576,
  "published_selection": 0.7374327692452669,
  "roi_qwk": 0.6927505726490435,
  "roi_macro_f1": 0.5793089399673257,
  "roi_macro_ap": 0.6361933785423153,
  "roi_selection": 0.6502345037285189
}

Training complete. Best robust_selection: 0.6976
Best checkpoint: /content/drive/MyDrive/Models/seresnext50_32x4d_yolo_384/2026-08-14_23-59-46_940080_UTC/best_model.pth


## Save metadata

In [7]:
metadata = {
    "model": "seresnext50_32x4d",
    "architecture": "seresnext50_32x4d_linear_gradcam",
    "input_size": INPUT_SIZE,
    "dataset": "paired_published_yolo_roi",
    "alternate_view_probability": ALTERNATE_VIEW_PROBABILITY,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "epochs": EPOCHS,
    "loss": "cross_entropy",
    "base_checkpoint": str(BASE_CHECKPOINT),
    "best_robust_selection": best_score,
    "checkpoint": str(RUN_DIR / "best_model.pth"),
    "train_samples": int((frame.split == "train").sum()),
    "val_samples":   int((frame.split == "val").sum()),
}
with open(RUN_DIR / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(f"Metadata saved: {RUN_DIR / 'metadata.json'}")
print(f"\n{'='*60}")
print("FINETUNING COMPLETE — proceed to notebook 04 for evaluation")
print(f"{'='*60}")
print(f"Best robust_selection (val): {best_score:.4f}")
print(f"Checkpoint: {RUN_DIR / 'best_model.pth'}")
print(f"Next step: Run notebook 04_evaluate")
print(f"{'='*60}")

Metadata saved: /content/drive/MyDrive/Models/seresnext50_32x4d_yolo_384/2026-08-14_23-59-46_940080_UTC/metadata.json

FINETUNING COMPLETE — proceed to notebook 04 for evaluation
Best robust_selection (val): 0.6976
Checkpoint: /content/drive/MyDrive/Models/seresnext50_32x4d_yolo_384/2026-08-14_23-59-46_940080_UTC/best_model.pth
Next step: Run notebook 04_evaluate
